# 🌳 Ternary-Bonsai-2-27B on Colab — llama.cpp + secured public API

Run cells top to bottom. At the end you get a **public HTTPS URL + API key + ready-to-paste OpenCode config**.

- Model: `prism-ml/Ternary-Bonsai-2-27B-gguf` (pick the packing in cell 1) — 100% in T4 VRAM
- Runner: PrismML-fork **prebuilt** `llama-server` (stock llama.cpp can't run the PQ2_0 / PTQ1_0 packings)
- Served model id: `bonsai-2-27b` · Context: **128K** (quantized KV cache) · API key: `ollama` by default

### Speed: what actually moves tok/s on a T4
Decode is memory-bandwidth bound: **tok/s ceiling ≈ 320 GB/s ÷ weight size** (7.6 GB → ~42, 5.95 GB → ~54). In practice expect 65–75% of that.
1. **Smaller packing:** `PTQ1_0` (5.95 GB) reads fewer bytes per token than `PQ2_0` (7.21 GB) or `Q2_0` (7.63 GB). Change `PACKING` in cell 1 and compare in cell 5.
2. **Less thinking:** `REASONING = 'medium'` makes the model think much less (about as accurate at moderate limits) so answers finish sooner.
3. **No wasted work:** single slot (`-np 1`), full GPU offload, flash attention on.
There is no speculative-decoding drafter for the 27B (PrismML ships MTP/DFlash drafters for 9B/4B only), so it is not used.

> ⚠️ Keep the tab open (free Colab disconnects on idle). Tunnel URL changes each run.

In [ ]:
# 1) Check GPU + settings (edit these)
import subprocess, shutil, secrets, json

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
assert gpu, 'No GPU found: Runtime > Change runtime type > T4 GPU'
print('GPU:', gpu)
g = shutil.disk_usage('/')
print(f'disk: {g.free/1e9:.0f} GB free (need ~15 GB)')

CFG = {
    'CTX': 131072,       # 128K context (model native is 262K)
    'KV': 'q8_0',        # KV-cache precision: q8_0 (~4.3 GB at 128K); auto-falls back to q4_0 if VRAM runs out
    'PACKING': 'PQ2_0',  # weights: 'PQ2_0' (7.21 GB) | 'PTQ1_0' (5.95 GB, fewest bytes/token: try for max speed) | 'Q2_0' (7.63 GB)
    'REASONING': 'medium',  # 'xhigh' (model default, slowest) | 'medium' (recommended for speed) | 'off' (no thinking)
    'FA': 'on',          # flash attention: 'on' (PrismML default) or 'off' (one community report says off was faster on V100: A/B it)
    'API_KEY': 'ollama', # what OpenCode / your .bat sends. Use secrets.token_urlsafe(16) for a private random key
    'PORT': 11434,
    'EXTRA': [],         # any extra llama-server flags, e.g. ['--no-webui']
}
json.dump(CFG, open('/content/config.json', 'w'))
print('config saved, API key =', CFG['API_KEY'])

In [ ]:
# 2) Download prebuilt llama-server (PrismML fork) + CUDA libs + fix the driver library path. No compile.
import subprocess, re, glob, os, site, json

CUDA_BUILD = '12.4'      # most compatible build; try '12.8' if you want to experiment
out = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True).stdout
m = re.search(r'Driver Version:\s*(\d+)\.', out)
major = int(m.group(1)) if m else 0

TAG = 'prism-b12128-d7b2896'
FORK = f'https://github.com/PrismML-Eng/llama.cpp/releases/download/{TAG}/'
STOCK = 'https://github.com/ggml-org/llama.cpp/releases/download/b12359/'
asset = f'llama-prism-b12128-d7b2896-bin-linux-cuda-{CUDA_BUILD}-x64.tar.gz'
print(f'NVIDIA driver {major}.x -> {asset}')

subprocess.run(f'curl -fsSL -o /content/{asset} {FORK}{asset}', shell=True, check=True)
subprocess.run(f'rm -rf /content/pkg && mkdir -p /content/pkg && tar -xzf /content/{asset} -C /content/pkg', shell=True, check=True)
found = glob.glob('/content/pkg/**/llama-server', recursive=True)
assert found, 'llama-server not found in archive'
found = found[0]
os.chmod(found, 0o755)

# CUDA runtime libs (cudart + cuBLAS). Bundle first, pip as a second source.
r = subprocess.run(f'curl -fsSL -o /content/cudart.tar.gz {STOCK}cudart-llama-b12359-bin-ubuntu-cuda-12.8-x64.tar.gz', shell=True)
subprocess.run('rm -rf /content/stock && mkdir -p /content/stock', shell=True)
if r.returncode == 0:
    subprocess.run('tar -xzf /content/cudart.tar.gz -C /content/stock', shell=True, check=True)
else:
    print('cudart bundle download failed, relying on pip CUDA 12 libs')
subprocess.run('pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12', shell=True, check=True)

# THE KEY FIX: the NVIDIA *driver* library (libcuda.so.1) lives in /usr/lib64-nvidia on Colab.
# Without it on LD_LIBRARY_PATH the CUDA backend fails to load and the server silently runs on CPU.
drv = glob.glob('/usr/lib64-nvidia/libcuda.so*') + glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*') \
    + glob.glob('/usr/local/nvidia/lib64/libcuda.so*')
assert drv, 'libcuda.so.1 not found: runtime has no working GPU (Runtime > Change runtime type > T4)'

dirs = [os.path.dirname(x) for x in glob.glob('/content/pkg/**/lib*.so*', recursive=True)]
dirs += [os.path.dirname(x) for x in glob.glob('/content/stock/**/lib*.so*', recursive=True)]
for sd in site.getsitepackages():
    dirs += [os.path.dirname(x) for x in glob.glob(os.path.join(sd, 'nvidia', '*', 'lib', 'lib*.so*'))]
dirs += [os.path.dirname(x) for x in drv]
dirs += [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p]
paths = list(dict.fromkeys(dirs))   # de-duplicate, keep order
envline = 'LD_LIBRARY_PATH=' + ':'.join(paths)
open('/content/llama_bin.txt', 'w').write(found)
open('/content/llama_env.txt', 'w').write(envline)
print('llama-server:', found)

# Verify the CUDA backend can load and sees the GPU
d = os.path.dirname(found)
bad = False
for so in glob.glob(d + '/libggml-cuda*'):
    missing = subprocess.run(f'{envline} ldd {so} | grep "not found"', shell=True, capture_output=True, text=True).stdout.strip()
    print(os.path.basename(so), '->', missing or 'all libraries found')
    bad |= bool(missing)
dev = subprocess.run(f'{envline} "{found}" --list-devices 2>&1 | head -10', shell=True, capture_output=True, text=True).stdout
print(dev)
assert not bad and 'none' not in dev.lower().split('available devices')[-1][:30], \
    'CUDA backend still cannot load: paste the output above for diagnosis'

In [ ]:
# 3) Download the model for the chosen packing (5.95-7.63 GB). Re-run to resume if interrupted.
import subprocess, os, json, urllib.request
CFG = json.load(open('/content/config.json'))
REPO = 'prism-ml/Ternary-Bonsai-2-27B-gguf'
fname = f"Ternary-Bonsai-2-27B-{CFG['PACKING']}.gguf"
MIN_GB = {'PQ2_0': 6.8, 'PTQ1_0': 5.6, 'Q2_0': 7.2}[CFG['PACKING']]

try:   # make sure the file exists in the repo, otherwise show what is available
    info = json.load(urllib.request.urlopen(f'https://huggingface.co/api/models/{REPO}', timeout=30))
    files = [x['rfilename'] for x in info.get('siblings', [])]
    if files and fname not in files:
        print('Available GGUF files:', [f for f in files if f.endswith('.gguf')])
        raise SystemExit(f'{fname} is not in {REPO}: pick another PACKING in cell 1')
except SystemExit:
    raise
except Exception as e:
    print('HF API check skipped:', e)

path = f'/content/{fname}'
subprocess.run(f'wget -c -q --show-progress -O {path} https://huggingface.co/{REPO}/resolve/main/{fname}', shell=True, check=True)
size = os.path.getsize(path) / 1e9
print(f'{fname}: {size:.2f} GB')
assert size > MIN_GB, 'Download incomplete, re-run this cell to resume'
CFG['MODEL'] = path
CFG['MODEL_GB'] = round(size, 2)
json.dump(CFG, open('/content/config.json', 'w'))

In [ ]:
# 4) Server helpers + start llama-server (everything on GPU, 128K context)
import subprocess, time, json, urllib.request, urllib.error, os, re, shlex

CFG = json.load(open('/content/config.json'))
LLAMA = open('/content/llama_bin.txt').read().strip()
ENV = open('/content/llama_env.txt').read().strip()
BASE = f"http://127.0.0.1:{CFG['PORT']}"
os.makedirs('/content/logs', exist_ok=True)
PROC = None

def save_cfg():
    json.dump(CFG, open('/content/config.json', 'w'))

def log_tail(n=2500):
    return open('/content/logs/llama-server.log', errors='ignore').read()[-n:]

def stop_server():
    global PROC
    if PROC and PROC.poll() is None:
        PROC.terminate()
        try: PROC.wait(timeout=15)
        except Exception: PROC.kill()
    subprocess.run(['pkill', '-x', 'llama-server'])   # -x = exact process name (never matches this shell)
    time.sleep(2)

def sampling_args():
    r = CFG.get('REASONING', 'medium')
    if r == 'off':    # PrismML's non-thinking (instruct) settings
        return ['--temp', '0.7', '--top-p', '0.8', '--top-k', '20', '--min-p', '0.0',
                '--presence-penalty', '1.5', '--reasoning-budget', '0']
    a = ['--temp', '1.0', '--top-p', '0.95', '--top-k', '20', '--min-p', '0.05']
    if r == 'medium':
        a += ['--chat-template-kwargs', json.dumps({'reasoning_effort': 'medium'})]
    return a

def _launch(extra, kv, wait):
    global PROC
    stop_server()
    args = [
        '-m', CFG['MODEL'], '--alias', 'bonsai-2-27b',
        '-ngl', '99', '-fa', CFG.get('FA', 'on'), '-c', str(CFG['CTX']),
        '-ctk', kv, '-ctv', kv,     # quantized KV cache so 128K fits next to the weights
        '-np', '1',                 # single slot: the whole context belongs to one session
        '-b', '2048', '-ub', '512',
        '--jinja',                  # chat template + native tool calling (needed by OpenCode)
        '--host', '127.0.0.1', '--port', str(CFG['PORT']),
    ] + sampling_args()
    if CFG.get('API_KEY'):          # empty string = no auth at all
        args += ['--api-key', CFG['API_KEY']]
    args += list(extra)
    cmd = f'exec env {shlex.quote(ENV)} {shlex.join([LLAMA] + args)} > /content/logs/llama-server.log 2>&1'
    PROC = subprocess.Popen(['bash', '-c', cmd])
    t0 = time.time()
    while time.time() - t0 < wait:
        time.sleep(2)
        if PROC.poll() is not None:     # real exit detection: fails in seconds, not minutes
            raise RuntimeError(f'llama-server exited (code {PROC.returncode}):\n' + log_tail())
        try:
            if urllib.request.urlopen(BASE + '/health', timeout=5).status == 200:
                CFG['CTX_USED'] = CFG['CTX']; save_cfg()
                print(f'server ready in {time.time()-t0:.0f}s  (ctx {CFG["CTX"]}, KV {kv}, FA {CFG.get("FA","on")}, reasoning {CFG.get("REASONING")})')
                return
        except Exception:
            pass
    raise RuntimeError('llama-server did not become healthy in time:\n' + log_tail())

def start_server(extra=(), wait=300):
    # try the configured KV precision first, then a smaller one if VRAM runs out
    kvs = list(dict.fromkeys([CFG.get('KV', 'q8_0'), 'q4_0']))
    for i, kv in enumerate(kvs):
        try:
            return _launch(extra, kv, wait)
        except RuntimeError as e:
            if i < len(kvs) - 1 and re.search(r'out of memory|cudaMalloc|failed to allocate', str(e), re.I):
                print(f'KV {kv} did not fit in VRAM, retrying with {kvs[i+1]}...')
                continue
            print(e); raise

def api(path, body, timeout=900, key=None):
    k = CFG.get('API_KEY') if key is None else key
    headers = {'Content-Type': 'application/json'}
    if k:
        headers['Authorization'] = 'Bearer ' + k
    req = urllib.request.Request(BASE + path, data=json.dumps(body).encode(), headers=headers)
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read())

start_server(CFG['EXTRA'])

In [ ]:
# 5) Smoke test + GPU check + speed benchmark
import subprocess

def gpu_used():
    o = subprocess.run('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits',
                       shell=True, capture_output=True, text=True).stdout.strip()
    return int(o.splitlines()[0])

used = gpu_used()
print(f'VRAM used: {used} MiB of 15360 (model + 128K KV cache)')
assert used > 5000, 'Model is NOT on the GPU (silent CPU fallback). Check log:\n' + log_tail(1500)

out = api('/v1/chat/completions', {'model': 'bonsai-2-27b', 'max_tokens': 400,
          'messages': [{'role': 'user', 'content': 'Reply with exactly: MODEL LOADED'}]})
print(out['choices'][0]['message']['content'][:200])

PROMPT = ('# Python 3\n# LRU cache with get/put and O(1) operations, plus unit tests\n'
          'from collections import OrderedDict\n\nclass LRUCache:\n')

def bench(n=256, runs=3):
    api('/completion', {'prompt': PROMPT, 'n_predict': 32, 'cache_prompt': False})  # warm-up
    speeds = []
    for _ in range(runs):
        r = api('/completion', {'prompt': PROMPT, 'n_predict': n, 'temperature': 0.2,
                                'cache_prompt': False, 'ignore_eos': True})
        speeds.append(r['timings']['predicted_per_second'])
    avg = sum(speeds) / len(speeds)
    print(f'decode: {avg:.1f} tok/s  (runs: {", ".join(f"{s:.1f}" for s in speeds)})')
    return avg

BASELINE = bench()
ceiling = 320 / CFG['MODEL_GB']     # T4 memory bandwidth (GB/s) / bytes read per token
print(f"T4 bandwidth ceiling for {CFG['PACKING']} ({CFG['MODEL_GB']} GB) = {ceiling:.0f} tok/s -> you are at {BASELINE/ceiling*100:.0f}% of it")
print('Want more? Set PACKING = "PTQ1_0" (smaller file) in cell 1, re-run cells 3-5, and compare this number.')

In [ ]:
# 6) Public tunnel (Cloudflare quick tunnel, no account) + ready-to-paste OpenCode config
import subprocess, time, re, json

CFG.update(json.load(open('/content/config.json')))
subprocess.run('pkill -f "[c]loudflared"; true', shell=True)
subprocess.run('curl -fsSL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared', shell=True, check=True)
subprocess.Popen(['bash', '-c',
    f"/content/cloudflared tunnel --url http://127.0.0.1:{CFG['PORT']} --no-autoupdate > /content/logs/tunnel.log 2>&1"])
url = None
for _ in range(40):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/logs/tunnel.log', errors='ignore').read())
    if m:
        url = m.group(0); break
if not url:
    print(open('/content/logs/tunnel.log', errors='ignore').read()[-800:])
    raise RuntimeError('tunnel did not come up')
time.sleep(5)  # let the edge register the tunnel
CFG['URL'] = url
json.dump(CFG, open('/content/config.json', 'w'))

def print_opencode():
    ctx = CFG.get('CTX_USED', CFG['CTX'])
    print('PUBLIC URL :', CFG['URL'])
    print('OpenAI base:', CFG['URL'] + '/v1')
    print('API key    :', CFG['API_KEY'] or '(none; any value works)')
    print('Model id   : bonsai-2-27b | context:', ctx, '\n')
    oc = {
        '$schema': 'https://opencode.ai/config.json',
        'provider': {'colab': {
            'npm': '@ai-sdk/openai-compatible', 'name': 'Bonsai 2 27B (Colab)',
            'options': {'baseURL': CFG['URL'] + '/v1', 'apiKey': CFG['API_KEY'] or 'none'},
            'models': {'bonsai-2-27b': {'name': 'Bonsai 2 27B',
                                        'limit': {'context': ctx, 'output': 16384}}}}}}
    print('--- paste into opencode.json (~/.config/opencode/opencode.json) ---')
    print(json.dumps(oc, indent=2))

print_opencode()

In [ ]:
# 7) Verify the public URL end-to-end
import json, urllib.request
CFG.update(json.load(open('/content/config.json')))
req = urllib.request.Request(CFG['URL'] + '/v1/chat/completions',
    data=json.dumps({'model': 'bonsai-2-27b', 'max_tokens': 400,
                     'messages': [{'role': 'user', 'content': 'Reply with exactly: API IS LIVE'}]}).encode(),
    headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ' + CFG['API_KEY'],
             'User-Agent': 'curl/8.0'})   # Cloudflare blocks the default Python user-agent
out = json.loads(urllib.request.urlopen(req, timeout=600).read())
print(out['choices'][0]['message']['content'][:300])
print('\nTimings:', {k: round(v, 1) for k, v in out.get('timings', {}).items() if k.endswith('per_second')})

In [ ]:
# 8) (Optional) FIX / CHANGE THE API KEY without losing the tunnel URL.
#    Use this when OpenCode gets "401 Unauthorized": the server only accepts the exact key it was started with.
#    Restarts the server with the FULL config (tool calling, 128K, speed flags) and the key you choose.
NEW_KEY = 'ollama'    # must equal apiKey in OpenCode. '' = disable authentication (anyone with the URL can use it)

import urllib.request, urllib.error
CFG['API_KEY'] = NEW_KEY
save_cfg()
start_server(CFG['EXTRA'])

def status(key, base=BASE, extra_headers=None):
    h = dict(extra_headers or {})
    if key is not None:
        h['Authorization'] = 'Bearer ' + key
    try:
        return urllib.request.urlopen(urllib.request.Request(base + '/v1/models', headers=h), timeout=30).status
    except urllib.error.HTTPError as e:
        return e.code

print('local, correct key :', status(NEW_KEY), '(expect 200)')
print('local, wrong key   :', status('wrong-key'), '(expect 401)' if NEW_KEY else '(no auth: 200 is expected)')
if CFG.get('URL'):
    print('public, correct key:', status(NEW_KEY, CFG['URL'], {'User-Agent': 'curl/8.0'}), '(expect 200)')
    print()
    print_opencode()
else:
    print('Tunnel not started yet: run cell 6 next.')

In [ ]:
# 9) (Optional) Keep-alive monitor: keeps the session busy and restarts server/tunnel if they die.
#    Leave this cell running; stop it with the square button.
import time, subprocess, urllib.request, json
CFG = json.load(open('/content/config.json'))
while True:
    time.sleep(60)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{CFG['PORT']}/health", timeout=10)
    except Exception:
        print('server down, restarting...'); start_server(CFG['EXTRA'])
    if subprocess.run('pgrep -f "[c]loudflared"', shell=True, capture_output=True).returncode != 0:
        print('tunnel down: re-run cell 6 (URL will change)'); break

## 🔌 Connect OpenCode (or any OpenAI client)
Cell 6 prints a complete `opencode.json` block with your URL and API key (default `ollama`). Merge it into your OpenCode config and pick **Bonsai 2 27B (Colab)**.
Other clients: `base_url = <URL>/v1`, `api_key = <printed key>`, model `bonsai-2-27b`.

### Notes
- New VM / restart: re-run cells 1→7 (binary ~2 min, model ~2-3 min). Cell 8 only if you hit 401 errors. New tunnel URL each time.
- 401 errors = key mismatch: OpenCode's `apiKey` must equal the key in cell 1 (or cell 8).
- Max speed: set `PACKING = 'PTQ1_0'` and `REASONING = 'medium'` (or `'off'`) in cell 1. Re-run cells 3-5 to compare tok/s.
- 128K context fits via a quantized KV cache (FP16 would need ~8 GB). A T4 processes long prompts slowly, and decode speed drops as the context fills, so use `/compact` and keep sessions well under 128K.
- If tok/s is low, check `nvidia-smi` shows ~8+ GB used (not 0 = CPU fallback).